---
title: "DRG Cleaning v2"

author: "Carlos Resurreccion"

date: "2024-10-21"

---


# !!! MAKE SURE YOU RESTART THE (JUPYTER) R KERNEL BEFORE PROCEEDING !!!


# Parameters

Change which year to process in
`~/drg-pipeline/data-cleaning/cache/year_to_load.txt`

Change main GLOBAL (i.e. across all scripts) parameters in 
`~/drg-pipeline/data-cleaning/00a-parameters.r`


Change seldom touched parameters in
`~/drg-pipeline/data-cleaning/r_scripts_v2/0.1.0.params_fpaths.R`

In [ ]:
source("~/drg-pipeline/data-cleaning/00a-parameters.r")


# Libraries


In [ ]:
# Update the grouper
system("git submodule update --init --recursive")

# List required packages
required_packages <- c(
  "data.table", # Fast data manipulation
  "here", # Simplifies file path management
  "tictoc", # Timing code execution
  "stringr", # String manipulation
  "stringi", # Unicode string processing
  "lubridate", # Date-time handling
  "profvis", # Profiling R code
  "hash", # Hashing utility
  "future", # Parallel processing
  "future.apply", # Parallelized apply functions
  "knitr", # Dynamic report generation
  "htmlwidgets", # Interactive HTML widgets
  "parallelly", # Advanced parallel computing
  "stringdist", # String distance calculations
  "parallel", # Base parallel computing
  "reticulate", # Interface to Python
  "bigrquery", # BigQuery client
  "jsonlite", # JSON parsing
  "googleCloudStorageR", # Google Cloud Storage access
  "haven", # Read/write Stata, SPSS, SAS files
  "fst", # Fast serialization
  "httr", # HTTP requests
  "ggplot2", # Data visualization
  "rmarkdown", # Dynamic markdown documents
  "digest", # Create cryptographic hashes
  "base64enc", # Base64 encoding/decoding
  "arrow", # Apache Arrow for fast data storage
  "tidyverse" # Collection of data science packages
)

github_packages <- c(
  "r-lib/styler" # Code formatting
)

# Installation commands (commented out, for reference)
# invisible(lapply(required_packages, function(pkg) if (!require(pkg, character.only = TRUE)) install.packages(pkg)))
# invisible(lapply(github_packages, function(repo) if (!require(basename(repo), character.only = TRUE)) remotes::install_github(repo)))

# Load packages (assumes they are already installed)
invisible(lapply(required_packages, library, character.only = TRUE))
invisible(lapply(basename(github_packages), library, character.only = TRUE))


# R Scripts


In [ ]:
year_to_load <- 2018

# Source each file sequentially
for (file in list.files(here::here("data-cleaning/r_scripts_v2"), pattern = "\\.R$", full.names = TRUE)) invisible(source(file))

message(year_to_load)


# Data Cleaning Proper


# Load Mapping Data


In [ ]:
# Enable caching and printing options for mapping data
to_use_cache <- TRUE # Use .rds cache files to speed up processing
to_print_mapping_data <- FALSE # Print mapping data tables if enabled

# Function to load data from cache if available, otherwise query from BigQuery
load_or_query <- function(query, var_name) {
  rds_path <- here(cache_path, "mapping", paste0(var_name, ".rds"))
  if (to_use_cache && file.exists(rds_path)) {
    return(readRDS(rds_path))
  } # Load cached data if exists
  dt <- query_bq_to_dt(query) # Query data from BigQuery
  saveRDS(dt, rds_path) # Cache the queried data
  return(dt)
}

# Function to print all rows of a data.table if printing is enabled
if (to_print_mapping_data) print_all <- function(dt, title) print(dt, nrow = Inf)

# Query and load procedure table (contains medical procedures and classifications)
proc <- load_or_query(paste0("SELECT * FROM `", gcp_proj, ".grouper_v5.proc`"), "proc")
proc[, CODE := as.character(CODE)] # Ensure CODE is stored as character

# Query and load RVS to ICD-9 mapping table (links procedure codes for billing)
rvs_icd9 <- load_or_query(paste0("SELECT * FROM `", gcp_proj, ".phic_libraries.acr_rvs_map`"), "rvs_icd9")
rvs_icd9 <- rvs_icd9[, .(rvs = as.character(rvs), icd9cm = as.character(as.numeric(icd9cm) * 100))]

# Merge RVS-ICD9 mapping with procedure table to identify DRG-related codes
rvs_icd9 <- merge(rvs_icd9, proc[, .(CODE, DRGUSE)], by.x = "icd9cm", by.y = "CODE", all.x = TRUE)
rvs_icd9 <- rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE][!is.na(rvs) & !is.na(icd9cm), -"DRGUSE"]

# Load RVS codes and descriptions (includes relative value units for medical billing)
acr_rvs <- load_or_query(paste0("SELECT * FROM `", gcp_proj, ".phic_libraries.acr_procedure`"), "acr_rvs")

# Load ICD-10 table for DRG classification (used in medical coding and billing)
tdrg_icd10 <- load_or_query(paste0("SELECT * FROM `", gcp_proj, ".grouper_v5.i10`"), "tdrg_icd10")
setkey(tdrg_icd10, "CODE") # Optimize lookups by setting CODE as key
acc_pdx <- unique(tdrg_icd10[ACCPDX == "Y", CODE]) # Extract valid primary diagnoses

# Create an environment for quick lookup of accepted primary diagnoses
acc_pdx_env <- new.env(hash = TRUE)
for (code in acc_pdx) assign(code, TRUE, envir = acc_pdx_env)

# Load Philippine ICD-10 table (local disease classifications and medical codes)
phl_icd10 <- load_or_query(paste0("SELECT * FROM `", gcp_proj, ".icd.phl_icd10`"), "phl_icd10")

# Extract neoplasm-related codes from the Philippine ICD-10 table
neoplasms_dt_actual <- as.data.table(phl_icd10[grepl("/", icd10), .(icd10)][, icd10 := sapply(strsplit(icd10, ","), trimws)])

# Load expanded ICD-10 table with validation flags
i10vx <- load_or_query(paste0("SELECT * FROM `", gcp_proj, ".grouper_v5.i10vx`"), "i10vx")
setkey(i10vx, "code") # Optimize lookups
acc_icd_set <- unique(i10vx[, code]) # Extract unique validated ICD-10 codes

# Load healthcare institution data (hospital and clinic listings)
hci <- load_or_query(paste0("SELECT * FROM `", gcp_proj, ".hci.temp_hci`"), "hci")

# Function to create a fast lookup environment from a vector of values
create_env_from_vector <- function(vec) {
  env <- new.env(parent = emptyenv())
  list2env(setNames(as.list(rep(TRUE, length(vec))), vec), envir = env)
  return(env)
}

# Create lookup environments for frequently accessed datasets
proc_env <- create_env_from_vector(proc$CODE)
rvs_env <- create_env_from_vector(rvs_icd9$rvs)
acr_rvs_env <- create_env_from_vector(acr_rvs$rvs)
acc_pdx_env <- create_env_from_vector(acc_pdx)
phl_icd10_env <- create_env_from_vector(phl_icd10$icd10)
acc_icd_env <- create_env_from_vector(i10vx$code)
hci_env <- create_env_from_vector(hci$id_hci)

# Save data tables for debugging if enabled
if (to_print_mapping_data) {
  options(max.print = 999999)
  output_file <- here(debug_path, "mapping_data.txt")
  sink(output_file) # Redirect output to file
  print(proc, nrow = Inf)
  print(rvs_icd9, nrow = Inf)
  print(acr_rvs, nrow = Inf)
  print(tdrg_icd10, nrow = Inf)
  print(acc_pdx, nrow = Inf)
  print(phl_icd10, nrow = Inf)
  print(neoplasms_dt_actual, nrow = Inf)
  print(i10vx, nrow = Inf)
  print(acc_icd_set, nrow = Inf)
  print(hci, nrow = Inf)
  sink()
  options(max.print = 1000)
}


Part 2: Main Data Cleaning Loop


In [ ]:
# Step 5: Loop through each part and process the partial files
# saveWidget(profvis({
for (loop_part in 1:split_parts) {
  # loop_part <- 1
  start_time <- Sys.time() # Record start time for processing
  # Step 7: Read the appropriate file (sample or full)
  cat(paste0("\rStart reading part ", loop_part, " of ", split_parts))
  flush.console()
  read_result <- read_appropriate_file(loop_part)
  # The data to process
  read_in_dt <- read_result$read_result_dt

  cat(paste0("\rFinished reading part ", loop_part, " of ", split_parts))
  flush.console()

  cat(paste0("\rStart chunking part ", loop_part, " of ", split_parts))
  flush.console()
  # Step 8: Split the data into chunks for parallel processing
  chunk_size <- ceiling(nrow(read_in_dt) / nthreads)
  chunks <- split(
    read_in_dt,
    rep(
      1:nthreads,
      each = chunk_size,
      length.out = nrow(read_in_dt)
    )
  )
  cat(paste0("\rFinished chunking part ", loop_part, " of ", split_parts))
  flush.console()

  cat(paste0("\rStart processing part ", loop_part, " of ", split_parts))
  flush.console()
  # Step 9: Apply parallel processing
  # See function(s) before the loop
  if (to_parallel) {
    parallel_results <- mclapply(
      chunks, process_chunk,
      mc.cores = nthreads
    )
  } else {
    if (!to_debug) {
      parallel_results <- lapply(chunks, process_chunk)
    } else {
      parallel_results <- list(process_chunk(chunks[[1]]))
    }
  }

  rbound_dt <- rbindlist(parallel_results)

  summarized_dt <- rbound_dt # Store the summarized data

  # Step 12: Write processed data to checkpoint file if required
  if (to_write) {
    saveRDS(
      summarized_dt, here(checkpoint_1_path, paste0(
        checkpoint_1_prefix, year_to_load, suffix,
        "part_", sprintf("%02d", loop_part), "_of_", split_parts, ".rds"
      )),
      compress = TRUE
    )
  }

  # Step 13: Collect summaries for each part
  processing_times[[loop_part]] <- as.numeric(difftime(Sys.time(),
    start_time,
    units = "secs"
  ))

  # Step 14: Update status and ETA
  print_status_update(loop_part, split_parts, processing_times, "clean")

  if (loop_part == 1) dim_dt <- dim(summarized_dt)

  nrow_end[[loop_part]] <- nrow(summarized_dt)
  # Step 15: Clean up memory after processing each part
  rm(read_in_dt, rbound_dt, summarized_dt)
  invisible(gc())
}
# }), profvis_fpath)


Part 3: Merge Partial Outputs & Print Checks and Summaries


In [ ]:
# Step 2: Combine all parts into a master data table
master_dt_list <- parallel::mclapply(1:split_parts, function(read_part) {
  cat(paste("\rStarted reading part", read_part))
  flush.console()
  return_dt <- readRDS(here(checkpoint_1_path, paste0(
    checkpoint_1_prefix, year_to_load, suffix,
    "part_", sprintf("%02d", read_part), "_of_", split_parts, ".rds"
  )))
  # message(colnames(return_dt))
  cat(paste("\rFinished reading prt", read_part))
  flush.console()
  return(return_dt)
}, mc.cores = nthreads)
message("Commencing rbindlist")
master_dt <- rbindlist(master_dt_list, fill = TRUE)
rm(master_dt_list)
invisible(gc())
message("Finished rbindlist")
# Step 1: Validate row counts across parts
# Initialize variable to track total row counts across parts
total_start_rows <- 0
total_end_rows <- 0

for (nrow_part in 1:split_parts) {
  # Sum up row counts for each part
  total_start_rows <- total_start_rows + nrow_start[[nrow_part]]
  total_end_rows <- total_end_rows + nrow_end[[nrow_part]]

  # Check if rows match for each part
  if (nrow_start[[nrow_part]] != nrow_end[[nrow_part]]) {
    warning(
      "WARNING: Row Count Mismatch! Part ", nrow_part,
      " has ", nrow_start[[nrow_part]], " starting rows and ",
      nrow_end[[nrow_part]], " ending rows\n"
    )
    stop("ERROR: Row Count Mismatch")
  }
}

# Check if the total rows match
if (if (to_sample) total_rows / sample_size_divisor else total_rows == nrow(master_dt)) {
  message("\nRow Counts Match for All Parts and Sum to Total Rows\n")
} else {
  stop("ERROR: Total Row Count Mismatch")
}

# Step 3: Save the combined master data table
if (to_write) {
  message("Commencing saveRDS")
  saveRDS(master_dt, here(
    checkpoint_2_path, paste0(
      checkpoint_2_prefix, year_to_load, suffix, ".rds"
    )
  ), compress = TRUE)
  message("Finished saveRDS")
}


Part 4: Save Output as .RDS


In [ ]:
if (exists("master_dt")) {
  message("master_dt exists, making a copy and deleting it")
  result <- data.table::copy(master_dt)
  rm(master_dt)
  invisible(gc())
  message("copied master_dt to result, deleted master_dt")
} else {
  message(paste0("master_dt doesn't exist, reading ", paste0(
    checkpoint_2_prefix, year_to_load, suffix, ".rds"
  )))
  result <- readRDS(here(
    checkpoint_2_path, paste0(
      checkpoint_2_prefix, year_to_load, suffix, ".rds"
    )
  ))
  invisible(gc())
  message(paste0("finished reading ", paste0(
    checkpoint_2_prefix, year_to_load, suffix, ".rds"
  )))
}

message(paste0("Saving ", paste0(checkpoint_2_prefix, year_to_load, suffix, "prefinal", ".rds")))
saveRDS(result, here(
  checkpoint_2_path,
  paste0(checkpoint_2_prefix, year_to_load, suffix, "prefinal", ".rds")
), compress = TRUE)
message(paste0("Finished saving ", paste0(checkpoint_2_prefix, year_to_load, suffix, "prefinal", ".rds")))


In [8]:
data <- readRDS("/home/resurreccion_cmc/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_2_master_clean_claims/checkpoint_2_claims_2018_sampled_625_prefinal.rds")
fwrite(data, "test.csv")


# Post Cleaning Steps

Check if all pdx are in acc_pdx

In [ ]:
if (to_post_cleaning_checks) {
  # Ensure acc_pdx is a set (i.e., unique values for faster lookup)
  acc_pdx_set <- unique(acc_pdx)

  dt <- readRDS(here(
    checkpoint_2_path,
    paste0(checkpoint_2_prefix, year_to_load, suffix, "prefinal", ".rds")
  ))

  str(dt)

  # Find entries in dt$clin_pdx that are not in acc_pdx_set
  not_in_acc_pdx <- dt$clin_pdx[!dt$clin_pdx %in% acc_pdx_set & !is.na(dt$clin_pdx)]

  # Check if all clin_pdx entries are in acc_pdx_set
  all_in_acc_pdx <- length(not_in_acc_pdx) == 0

  # Output the result
  if (all_in_acc_pdx) {
    message("All entries in dt$clin_pdx are in acc_pdx.")
  } else {
    message("Not all entries in dt$clin_pdx are in acc_pdx. Entries not in acc_pdx are:")
    print(unique(not_in_acc_pdx)) # Print unique entries not in acc_pdx
  }
  rm(all_in_acc_pdx, not_in_acc_pdx)
  invisible(gc())
}


Check breakdown histogram of pdx code sources

In [ ]:
if (to_post_cleaning_checks) {
  # Assuming dt is your data.table
  # Calculate counts per category and total count
  count_data <- dt[, .N, by = clin_pdx_source]
  setorder(count_data, clin_pdx_source)
  count_data[, clin_pdx_source := factor(clin_pdx_source, levels = c(1, 2, 3, 6, 99))]

  # Calculate total count
  total_count <- sum(count_data$N)

  # Plot histogram with counts on top of each bar and total count below
  ggplot(count_data, aes(x = clin_pdx_source, y = N)) +
    geom_bar(stat = "identity", fill = "skyblue", color = "black") +
    labs(title = "Histogram of clin_pdx_source", x = "clin_pdx_source", y = "Count") +
    theme_minimal() +
    scale_x_discrete(drop = FALSE) + # Ensures all categories are shown
    geom_text(aes(label = N), vjust = -0.5) + # Display count above each bar
    annotate("text", x = Inf, y = -Inf, label = paste("Total N =", total_count), hjust = 1.1, vjust = -1.5) # Display total count below
  rm(count_data, total_count)
  invisible(gc())
}


Check contents of c1, c2, clin_sdx, clin_pdx, clin_proc

In [ ]:
if (to_post_cleaning_checks) {
  # Capture the combined structure output for each column into a single text variable
  output <- capture.output({
    cat("Structure of non-empty elements in each specified column:\n\n")
    cat("dt\n")
    str(dt)

    cat("Structure of non-empty elements in each specified column:\n\n")
    cat("c1:\n")
    str(dt[!is.na(c1) & sapply(c1, function(x) length(x) > 0 && any(nzchar(x)))]$c1)

    cat("\nc2:\n")
    str(dt[!is.na(c2) & sapply(c2, function(x) length(x) > 0 && any(nzchar(x)))]$c2)

    cat("\nclin_sdx:\n")
    str(dt[!is.na(clin_sdx) & sapply(clin_sdx, function(x) length(x) > 0 && any(nzchar(x)))]$clin_sdx)

    cat("\nclin_pdx:\n")
    str(dt[!is.na(clin_pdx) & sapply(clin_pdx, function(x) length(x) > 0 && any(nzchar(x)))]$clin_pdx)

    cat("\nclin_proc:\n")
    str(dt[!is.na(clin_proc) & sapply(clin_proc, function(x) length(x) > 0 && any(nzchar(x)))]$clin_proc)
  })

  # Print the output as a single message or save it to a file
  cat(paste(output, collapse = "\n"))
  rm(output)
  invisible(gc())
}


Check non-empty clin_rvs rows that have a matching icd9cm code

In [ ]:
if (to_post_cleaning_checks) {
  # Flatten the list, get unique values, and omit NA
  unique_values <- unique(unlist(dt$clin_rvs))
  unique_values <- unique_values[!is.na(unique_values) & unique_values != "NA"]

  # Filter values that are also in rvs_icd9$rvs
  matched_values <- unique_values[unique_values %in% rvs_icd9$rvs]

  # Print the matched values, separated by line breaks
  cat(paste(matched_values, collapse = "\n"))
  rm(unique_values, matched_values)
  invisible(gc())
}


Check non-empty clin_proc rows


In [ ]:
if (to_post_cleaning_checks) {
  # Assuming `dt` is your data.table
  # Filter rows where clin_proc is not NA and does not contain character(0) or empty strings
  non_empty_clin_proc_rows <- dt[!is.na(clin_proc) & sapply(clin_proc, function(x) length(x) > 0 && any(nzchar(x)))]

  # Print the result
  print(non_empty_clin_proc_rows)
  rm(non_empty_clin_proc_rows)
  invisible(gc())
}


# Post Cleaning Steps


# Part 2: Final Checks


Part 2A: Final Checks - Checking for Exponential Form Numbers


In [ ]:
if (to_post_cleaning_checks) {
  result <- readRDS(here(
    checkpoint_2_path,
    paste0(checkpoint_2_prefix, year_to_load, suffix, "prefinal", ".rds")
  ))

  setkey(result, NULL) # Removes any existing key
  print(result[id_series %like% "e"])
  print(result[id_pin %like% "e"])
  print(result[id_hci %like% "e"])

  # Flatten clin_sdx and check if "A" is present in any element
  flattened_clin_sdx <- unlist(result$clin_sdx, use.names = FALSE, recursive = TRUE)

  # Check if "A" is in any element using %chin% (fast for exact matches)
  if ("A" %chin% flattened_clin_sdx) {
    cat("Found 'A' in clin_sdx\n")
    # Optionally, filter rows that contain "A" in clin_sdx and print progress every 100,000 rows
    rows_with_A <- result[sapply(result$clin_sdx, function(x) any("A" %chin% x))]

    for (i in seq_len(nrow(rows_with_A))) {
      print(rows_with_A[i])
    }
  } else {
    cat("No 'A' found in clin_sdx\n")
  }
}


Part 2B: Final Checks - Checking for Dates Before 1900


In [ ]:
if (to_post_cleaning_checks) {
  result <- readRDS(here(
    checkpoint_2_path,
    paste0(checkpoint_2_prefix, year_to_load, suffix, "prefinal", ".rds")
  ))

  date_cols <- c(
    "date_adm", "date_dis", "date_rec", "date_ref",
    "date_check", "pat_bdate", "date_ext"
  )

  # Find rows where any date column has a date before 1900-01-01
  rows_with_old_dates <- result[Reduce(`|`, lapply(
    .SD,
    function(x) x < as.Date("1900-01-01")
  )), .SDcols = date_cols]

  # Print the resulting rows
  print(rows_with_old_dates)
}


Part 2C: Final Checks - Dropping Duplicate Columns


In [ ]:
result <- readRDS(here(
  checkpoint_2_path,
  paste0(checkpoint_2_prefix, year_to_load, suffix, "prefinal", ".rds")
))
print(nrow(result))
# result <- result[clin_outpatient == FALSE, ] # DONT SUBSET OUTPATIENT CLAIMS
print(nrow(result))
result[, is_covid := {
  # Start with FALSE
  covid_found <- rep(FALSE, .N)

  # Check each condition sequentially, updating only rows not yet marked TRUE
  not_found <- !covid_found
  covid_found[not_found] <- clin_c1[not_found] %chin% covid_rvs

  not_found <- !covid_found
  covid_found[not_found] <- clin_c2[not_found] %chin% covid_rvs

  not_found <- !covid_found
  covid_found[not_found] <- c2[not_found] %chin% covid_rvs

  not_found <- !covid_found
  covid_found[not_found] <- c1[not_found] %chin% covid_rvs

  not_found <- !covid_found
  covid_found[not_found] <- sapply(clin_rvs[not_found], function(row) any(row %chin% covid_rvs))

  not_found <- !covid_found
  covid_found[not_found] <- sapply(clin_sdx[not_found], function(row) any(row %chin% covid_rvs))

  not_found <- !covid_found
  covid_found[not_found] <- sapply(clin_proc[not_found], function(row) any(row %chin% covid_rvs))

  # Return the result
  covid_found
}]

# result <- result[is_covid == FALSE, ] # DONT SUBSET COVID CLAIMS
print(nrow(result))
saveRDS(result, here(
  checkpoint_2_path,
  paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")
))


Part 2D: Final Checks - Checking Age Changes


In [ ]:
if (to_post_cleaning_checks) {
  before <- readRDS(here(
    checkpoint_2_path,
    paste0(checkpoint_2_prefix, year_to_load, suffix, ".rds")
  ))
  after <- readRDS(here(
    checkpoint_2_path,
    paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")
  ))

  # Ensure both data.tables have the same key columns for comparison
  setkey(before, id_series)
  setkey(after, id_series)

  # Identify rows where pat_age is different
  # between the two tables, handling NA values
  pat_age_diff_na <- before[after,
    on = .(id_series), nomatch = 0,
    # Explicitly name pat_age_before as coming from "before"
    .(id_series, pat_bdate,
      pat_age_before = x.pat_age,
      pat_age_after = i.pat_age
    ),
    by = .EACHI
  ]

  # Filter to show rows where one value is NA and
  # the other is not or the values are simply different
  pat_age_diff_na <- pat_age_diff_na[
    (is.na(pat_age_before) & !is.na(pat_age_after)) |
      (!is.na(pat_age_before) & is.na(pat_age_after)) |
      (pat_age_before != pat_age_after)
  ]

  # Print the differences
  cat("Rows where pat_age is NA in one table but
not in the other, or where the values differ:\n")
  print(pat_age_diff_na)

  rm(before, after)
  invisible(gc())
}


# BQ Upload


Prepare for BQ upload


In [ ]:
message("Reading final")
result <- readRDS(here(
  checkpoint_2_path,
  paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")
))
message("Finished reading final, commencing subsetting")
result <- result[, .(
  id_series, id_pin, id_hci, id_hcp, date_adm, time_adm,
  date_dis, time_dis, date_rec, date_ref, date_check, pat_type, pat_rel, pat_bdate,
  pat_age, pat_ageday, pat_sex, pat_bwt, pat_memcat_parent,
  pat_memcat_child, claim_status, claim_payout, claim_charge, is_covid,
  clin_discharge, clin_outpatient, clin_emergency, clin_acc,
  clin_c1, clin_c2, clin_sdx, clin_proc, clin_pdx, clin_pdx_source
)]
message("Finished subsetting, commencing saveRDS")
saveRDS(result, here(
  checkpoint_2_path,
  paste0(checkpoint_2_prefix, year_to_load, suffix, "final_subset_with_time", ".rds")
))
message("Finished saveRDS, commencing subsetting")
result <- result[, .(
  id_series, id_pin, id_hci, id_hcp, date_adm,
  date_dis, date_rec, date_ref, date_check, pat_type, pat_rel,
  pat_age, pat_ageday, pat_sex, pat_bwt, pat_memcat_parent,
  pat_memcat_child, claim_status, claim_payout, claim_charge, is_covid,
  clin_discharge, clin_outpatient, clin_emergency, clin_acc,
  clin_c1, clin_c2, clin_sdx, clin_proc, clin_pdx, clin_pdx_source
)]
message("Finished subsetting, commencing saveRDS")
saveRDS(result, here(
  checkpoint_2_path,
  paste0(checkpoint_2_prefix, year_to_load, suffix, "final_subset", ".rds")
))
message("Finished saveRDS")


In [19]:
result <- readRDS(here(
  checkpoint_2_path,
  paste0(checkpoint_2_prefix, year_to_load, suffix, "final_subset", ".rds")
))


In [20]:
if (to_bq) {
  if (!to_sample) bq_table <- paste0("claims_", year_to_load)

  # Check if the table should be dropped and replaced
  tryCatch(
    {
      bq_table_delete(bq_table(gcp_proj, bq_dataset, bq_table))
      message("Table dropped successfully.\n")
    },
    error = function(e) {
      # If the table does not exist, just continue
      if (grepl("Not found", e, ignore.case = TRUE)) {
        message("Table does not exist, nothing to drop.\n")
      } else {
        # If it's a different error, re-throw the error
        stop(e)
      }
    }
  )

  # Attempt to create the table
  tryCatch(
    {
      bq_table_create(
        bq_table(gcp_proj, bq_dataset, bq_table),
        fields = fromJSON(here(
          "data-cleaning/r_scripts_v2",
          "bq_schema_cleaning.json"
        ), simplifyDataFrame = FALSE)
      )
      message("Table created successfully.\n")
    },
    error = function(e) {
      # Check if the error message indicates that the table already exists
      if (grepl("already exists", e, ignore.case = TRUE)) {
        message("Table already exists. Skipping creation and upload.")
      } else {
        # If it's a different error, re-throw the error
        stop(e)
      }
    }
  )

  # Upload to BQ only if table is empty
  if (to_write) {
    chunk_size <- 250000 # Adjust the chunk size based on memory availability
    num_chunks <- ceiling(nrow(result) / chunk_size)

    for (i in seq_len(num_chunks)) {
      cat(paste("\rUploading chunk no.:", i))
      flush.console()
      chunk <- result[
        ((i - 1) * chunk_size + 1):min(i * chunk_size, nrow(result)),
      ]

      bq_table_upload(
        bq_table(gcp_proj, bq_dataset, bq_table),
        values = chunk,
        write_disposition = if (i == 1) "WRITE_EMPTY" else "WRITE_APPEND"
      )
      cat(paste("\rUploaded chunk no.:", i))
      flush.console()
    }
  }
}
